# Recruitment Screening and Interview Scheduling Pipeline

This demo runs the Semantic Kernel pipeline on three candidate sets. Each run prints the ordered agent/tool trace, the requirement evidence and score, the bias review, and any reserved interview slot. The agents use deterministic local functions, so no model credentials are needed.

In [ ]:
from pathlib import Path
import pprint
import sys

project_root = Path.cwd()
if not (project_root / 'pipeline').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from pipeline.coordinator import PipelineCoordinator

job = {
    'title': 'Data Analyst',
    'requirements': [
        {'name': 'SQL querying', 'kind': 'skill', 'expected': 'sql', 'weight': 2},
        {'name': 'Python', 'kind': 'skill', 'expected': 'python', 'weight': 1},
        {'name': 'Two years experience', 'kind': 'experience', 'expected': 2, 'weight': 1},
        {'name': 'Bachelor education', 'kind': 'education', 'expected': 'bachelor', 'weight': 1},
    ],
}

def print_result(result):
    print('MULTI-STEP TRACE')
    pprint.pp(result['trace'])
    print('RANKED SHORTLIST')
    for candidate in result['candidate_pool']:
        print({
            'rank': candidate['rank'],
            'candidate_id': candidate['candidate_id'],
            'score': candidate['score'],
            'requirement_evidence': candidate['requirement_results'],
            'rationale': candidate['rationale'],
            'bias_review': candidate['bias_review'],
            'interview_slot': candidate['interview_slot'],
        })

## Run 1: Full shortlist
The Coordinator sends the same in-memory candidate pool through parsing, matching, ranking, review, and scheduling.

In [ ]:
resumes_1 = [
    {'candidate_id': 'C-101', 'text': 'Skills: SQL, Python, Tableau. 4 years experience. BSc in Statistics.'},
    {'candidate_id': 'C-102', 'text': 'Skills: SQL, Excel. 1 year experience. Bachelor degree.'},
    {'candidate_id': 'C-103', 'text': 'Skills: Python, Excel. 3 years experience. MSc in Analytics.'},
]
slots_1 = [
    {'slot_id': 'INT-01', 'start': '2026-12-01T09:00:00'},
    {'slot_id': 'INT-02', 'start': '2026-12-01T10:00:00'},
]
result_1 = await PipelineCoordinator(slots_1).run(job, resumes_1, top_n=2)
print_result(result_1)

## Run 2: Availability-limited
The Scheduler receives the ranked pool and one available slot; remaining shortlisted candidates stay unscheduled.

In [ ]:
resumes_2 = [
    {'candidate_id': 'C-201', 'text': 'Skills: SQL, Python. 5 years experience. BSc.'},
    {'candidate_id': 'C-202', 'text': 'Skills: SQL, Excel. 3 years experience. BSc.'},
]
slots_2 = [{'slot_id': 'INT-03', 'start': '2026-12-02T13:00:00'}]
result_2 = await PipelineCoordinator(slots_2).run(job, resumes_2, top_n=2)
print_result(result_2)

## Run 3: Missing evidence and protected-attribute noise
The parser discards raw resume text after extracting job-related fields. Explicit protected-attribute text is not part of the shared pool or score. Missing experience evidence is reported rather than inferred from dates.

In [ ]:
resumes_3 = [
    {'candidate_id': 'C-301', 'text': 'Age: 28. Gender: female. Skills: SQL, Python. Employment dates: 2020-2024. BSc.'},
    {'candidate_id': 'C-302', 'text': 'Skills: Excel. 2 years experience. Associate degree.'},
]
result_3 = await PipelineCoordinator([]).run(job, resumes_3, top_n=2)
print_result(result_3)